In [1]:
import torch
from pathlib import Path
from main import load_dotenv
from data_loader.raw_dataloader import RawDataloader
from collections import Counter, defaultdict
from miditok import REMI

load_dotenv(Path("../.env"))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


/home/skynet/research/MusicGeneration/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

print("Torch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())

Torch: 2.13.0+cu130
CUDA build: 13.0
CUDA available: True
Device count: 1


In [3]:
dataloader = RawDataloader()
dataloader.prepare_dataset(representation="remi")
loader = dataloader.loader(load_music=dataloader.load_music)

In [4]:
tokenizer = REMI(params="/home/skynet/research/data/maestro/maestro-v3.0.0_cononical/remi/tokenizer.json")
vocab_size = len(tokenizer)

token_to_id = {
    token: token_id
    for token, token_id in tokenizer.vocab.items()
}

id_to_token = {
    token_id: token
    for token, token_id in tokenizer.vocab.items()
}

sequence = [token_to_id[token] for token in tokenizer.vocab.keys()]

/home/skynet/research/MusicGeneration/.venv/lib/python3.13/site-packages/miditok/tokenizations/remi.py:88: UserWarning: Attribute controls are not compatible with 'config.one_token_stream_for_programs' and multi-vocabulary tokenizers. Disabling them from the config.
  super().__init__(tokenizer_config, params)


In [5]:
import torch
import torch.nn as nn

class MusicRNN(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=64,
        hidden_size=128,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
        )

        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            batch_first=True,
        )

        self.output = nn.Linear(
            hidden_size,
            vocab_size,
        )

    def forward(self, x, hidden=None):
        x = self.embedding(x)

        x, hidden = self.rnn(
            x,
            hidden,
        )

        logits = self.output(x)

        return logits, hidden

In [6]:
import random

def iter_training_batches(
    song_loader,
    sequence_length=512,
    stride=256,
    batch_size=32,
    buffer_size=2048,
):
    buffer = []

    for song_batch in song_loader:
        for song in song_batch:
            ids = song["tokens"]

            for start in range(
                0,
                len(ids) - sequence_length,
                stride,
            ):
                chunk = ids[start:start + sequence_length + 1]

                x = chunk[:-1]
                y = chunk[1:]

                buffer.append((x, y))

                if len(buffer) >= buffer_size:
                    random.shuffle(buffer)

                    while len(buffer) >= batch_size:
                        yield [
                            buffer.pop()
                            for _ in range(batch_size)
                        ]

    random.shuffle(buffer)

    while len(buffer) >= batch_size:
        yield [
            buffer.pop()
            for _ in range(batch_size)
        ]

In [7]:
criterion = nn.CrossEntropyLoss()

model = MusicRNN(vocab_size=vocab_size, embedding_dim=64, hidden_size=128)
model.to(device)

x = torch.tensor(
    [sequence[:-1]],
    dtype=torch.long,
)

y = torch.tensor(
    [sequence[1:]],
    dtype=torch.long,
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
)

In [8]:
def train_batch(batch):
    x = torch.tensor(
        [sample[0] for sample in batch],
        dtype=torch.long,
        device=device,
    )

    y = torch.tensor(
        [sample[1] for sample in batch],
        dtype=torch.long,
        device=device,
    )

    optimizer.zero_grad()

    logits, _ = model(x)

    loss = criterion(
        logits.reshape(-1, vocab_size),
        y.reshape(-1),
    )

    loss.backward()
    optimizer.step()

    return loss.item()

In [9]:
from tqdm.auto import tqdm

In [10]:
from tqdm.auto import tqdm

num_epochs = 30

for epoch in range(num_epochs):
    progress_loader = tqdm(
        loader,
        total=len(loader),
        desc=f"Epoch {epoch + 1}/{num_epochs}",
        leave=True,
    )

    loss_sum = 0.0
    loss_count = 0

    for batch_idx, batch in enumerate(
        iter_training_batches(
            batch_size=32,
            sequence_length=256,
            stride=256,
            buffer_size=2048,
            song_loader=progress_loader,
        ),
        start=1,
    ):
        loss = train_batch(batch)

        loss_sum += loss
        loss_count += 1

        if batch_idx % 10 == 0:
            mean_loss = loss_sum / loss_count

            progress_loader.set_postfix(
                batch=batch_idx,
                loss=f"{loss:.4f}",
                mean_loss=f"{mean_loss:.4f}",
            )

Epoch 30/30: 100%|██████████| 31/31 [00:15<00:00,  2.06it/s, batch=3200, loss=1.9176, mean_loss=1.9025]
